<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M09/M09_Lab1_Gradio_Apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🖥️ M09 Lab 1 — Gradio Apps</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Install and use <b>Gradio</b> to build AI-powered web apps in minutes</li>
    <li>Build a <b>text analyzer</b> with sentiment, summary, and key topics</li>
    <li>Create a <b>chatbot UI</b> connected to OpenAI</li>
    <li>Build an <b>image describer</b> using GPT-4V (vision)</li>
    <li>Launch a <b>public link</b> to share your app with anyone</li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai gradio
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

client = setup_openai()

import gradio as gr
print(f"✅ Gradio version: {gr.__version__}")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Why Gradio?</h2>
</div>

You've built powerful AI pipelines with LangChain, agents, and RAG. But how do **non-technical users** interact with them? They need a **UI**.

**Gradio** lets you wrap any Python function in a web interface with just a few lines of code:

| Feature | Gradio | Streamlit | React |
|---------|--------|-----------|-------|
| Lines of code | 5-10 | 20-50 | 100+ |
| Runs in Colab | ✅ Yes | ❌ Needs workaround | ❌ No |
| Public sharing | ✅ One flag | ❌ Needs deploy | ❌ Needs deploy |
| Chat interface | ✅ Built-in | ✅ Built-in | 🔧 Manual |
| Best for | Demos, prototypes | Dashboards | Production apps |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ App 1: AI Text Analyzer</h2>
</div>

Our first app takes text input and returns AI-powered analysis: sentiment, summary, and key topics.

In [ ]:
# ============================================================
# 📝 App 1: Text Analyzer
# ============================================================

def analyze_text(text):
    """Analyze text using GPT — returns sentiment, summary, and key topics."""
    if not text.strip():
        return "Please enter some text to analyze."

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{
            "role": "user",
            "content": f"""Analyze the following text and return:
1. **Sentiment**: Positive, Negative, or Neutral (with confidence %)
2. **Summary**: 1-2 sentence summary
3. **Key Topics**: Top 3 topics as bullet points
4. **Tone**: Professional, Casual, Academic, etc.

Text: {text}"""
        }],
        temperature=0.3
    )
    return response.choices[0].message.content

# Create the Gradio interface
app1 = gr.Interface(
    fn=analyze_text,
    inputs=gr.Textbox(lines=6, placeholder="Paste any text here — news article, review, email..."),
    outputs=gr.Markdown(),
    title="🔍 AI Text Analyzer",
    description="Paste text and get instant AI analysis: sentiment, summary, key topics, and tone.",
    examples=[
        ["The new supply chain optimization tool has dramatically reduced our inventory costs by 34%. The team is thrilled with the results and management has approved expansion to all warehouses."],
        ["I'm extremely disappointed with the customer service. After waiting 45 minutes on hold, the representative was unhelpful and dismissive. I'm considering switching providers."],
    ]
)

app1.launch(share=True, height=600)

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Setting <code>share=True</code> generates a public URL (valid for 72 hours) that anyone can access — no deployment needed! Perfect for demos and hackathons. The URL routes through Gradio's servers to your Colab runtime.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ App 2: AI Chatbot</h2>
</div>

Gradio has a built-in `ChatInterface` that handles the full chat UI — message history, typing indicator, and clear button — all in ~10 lines.

In [ ]:
# ============================================================
# 💬 App 2: AI Chatbot with OpenAI
# ============================================================

def chat_with_ai(message, history):
    """Chat function for Gradio ChatInterface."""
    # Convert Gradio history format to OpenAI messages
    messages = [{"role": "system", "content": "You are a helpful AI assistant. Keep answers concise (2-3 sentences)."}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

app2 = gr.ChatInterface(
    fn=chat_with_ai,
    title="🤖 DADS 5250 AI Assistant",
    description="Ask me anything about GenAI, supply chain, data science, or course topics!",
    examples=[
        "What is RAG and why is it useful?",
        "Explain the bullwhip effect in 2 sentences",
        "What's the difference between LangChain and LangGraph?"
    ]
)

app2.launch(share=True, height=600)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ App 3: Image Describer (GPT-4V)</h2>
</div>

This app lets users upload an image, and GPT-4V describes it in detail. We encode the image as base64 and send it to the Vision API.

In [ ]:
# ============================================================
# 🖼️ App 3: Image Describer with GPT-4V
# ============================================================
import base64

def describe_image(image, question="Describe this image in detail."):
    """Send an image to GPT-4V and get a description."""
    if image is None:
        return "Please upload an image."

    # Convert PIL image to base64
    import io
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    b64_image = base64.b64encode(buffer.getvalue()).decode("utf-8")

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{b64_image}"
                }}
            ]
        }],
        max_tokens=500
    )
    return response.choices[0].message.content

app3 = gr.Interface(
    fn=describe_image,
    inputs=[
        gr.Image(type="pil", label="Upload an Image"),
        gr.Textbox(value="Describe this image in detail.", label="Your Question")
    ],
    outputs=gr.Markdown(label="AI Description"),
    title="🖼️ AI Image Describer",
    description="Upload any image and ask AI to describe it, read text from it (OCR), or analyze its content."
)

app3.launch(share=True, height=600)

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Each Gradio app follows the same pattern: <b>define a function</b> → <b>wrap with gr.Interface()</b> → <b>launch()</b>. The framework handles all the UI rendering, input parsing, and output formatting. You only write the AI logic.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What does `share=True` do when launching a Gradio app?",
        "options": [
            "Uploads the app to Hugging Face Spaces permanently",
            "Creates a temporary public URL (72 hours) that anyone can access",
            "Shares the source code of the app on GitHub",
            "Enables multiple users to edit the app simultaneously"
        ],
        "answer": 1,
        "explanation": "share=True generates a temporary public URL (valid for 72 hours) that tunnels through Gradio's servers to your running notebook. Anyone with the link can use your app without any deployment."
    },
    {
        "q": "What is the minimum code needed to create a Gradio app?",
        "options": [
            "A Python function, a React component, and a CSS file",
            "A Python function + gr.Interface(fn, inputs, outputs) + .launch()",
            "A Flask server, HTML template, and JavaScript frontend",
            "A Docker container with a Python function inside"
        ],
        "answer": 1,
        "explanation": "Gradio only needs three things: a Python function that processes input, a gr.Interface() that defines the UI components, and a .launch() call. That's typically 5-10 lines of code total."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build Your Own Gradio App

Build an **AI Resume Reviewer** that takes resume text as input and returns structured feedback. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: AI Resume Reviewer
# ============================================================
# Replace each ----- with the correct value

def review_resume(resume_text):
    """AI-powered resume reviewer."""
    if not resume_text.strip():
        return "Please paste your resume text."

    response = client.chat.completions.create(
        model="-----",              # Which model?
        messages=[{
            "role": "user",
            "content": f"""Review this resume and provide:
1. **Overall Score**: X/10
2. **Strengths**: Top 3 strengths
3. **Improvements**: Top 3 areas to improve
4. **Missing Keywords**: Suggest 5 keywords to add for ATS optimization
5. **Rewritten Summary**: A better professional summary (2-3 sentences)

Resume:
{resume_text}"""
        }],
        temperature=-----          # Low or high temperature for this task?
    )
    return response.choices[0].message.content

app_exercise = gr.Interface(
    fn=-----,                      # Which function?
    inputs=gr.Textbox(lines=10, placeholder="Paste your resume text here..."),
    outputs=gr.-----(),            # What output component for formatted text?
    title="📄 AI Resume Reviewer",
    description="Paste your resume and get instant AI feedback with scores, strengths, and improvements."
)

app_exercise.launch(share=True, height=600)

**📝 Your Observations:** *(double-click to edit this cell)*

1. How long did it take to build a functional AI app with Gradio? _[Your answer]_

2. What other AI-powered apps could you build with this pattern? _[Your answer]_

3. What are the limitations of Gradio for production applications? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try building these apps on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>AI Email Writer:</b> Input: context + tone → Output: professional email draft</li>
    <li><b>Code Explainer:</b> Input: code snippet → Output: line-by-line explanation</li>
    <li><b>Multi-tab app:</b> Use <code>gr.TabbedInterface</code> to combine all 3 apps into one dashboard</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built 3 AI-powered Gradio apps — text analyzer, chatbot, and image describer.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">gr.Interface(fn, inputs, outputs)</code> — wrap any function in a web UI</li>
      <li><code style="color: #90caf9;">gr.ChatInterface(fn)</code> — instant chatbot with history</li>
      <li><code style="color: #90caf9;">share=True</code> — public URL for instant demos</li>
      <li>Gradio is ideal for <b>prototypes and hackathons</b> — build in minutes, not days</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M09 Lab 2: Streamlit Dashboard</p>
</div>